# Video Generation from Scratch — Student Lab

Train a small MNIST generator from random initialization, then generate your own images and sampling animation. This is the foundations lab: the animations show image sampling, and video generation is a later part of the series.

[Step-by-step guide](https://clguo.github.io/tutorials/video-generation/foundations/start-here.html) · [Theory](https://clguo.github.io/tutorials/video-generation/foundations/) · [Source](https://github.com/clguo/clguo.github.io/tree/main/tutorials/video-generation/foundations)

Use a CUDA GPU runtime for training. In Colab, select a GPU hardware accelerator in the runtime settings. Run one cell at a time and read the output before continuing. Download your `results/` and `outputs/` files before a temporary notebook runtime is reset.


## 1. Get the code
If this notebook is already running from the extracted code folder, this cell keeps that folder. Otherwise it downloads the publication bundle and enters a new `mnist-tutorial` directory.


In [ ]:
from pathlib import Path
import os
import urllib.request
import zipfile

if not Path("unified/train.py").is_file():
    lab_root = Path("mnist-tutorial")
    lab_root.mkdir(exist_ok=True)
    lab_archive = lab_root / "materials.zip"
    urllib.request.urlretrieve(
        "https://clguo.github.io/tutorials/video-generation/foundations/mnist-generative-models-unified.zip",
        lab_archive,
    )
    with zipfile.ZipFile(lab_archive) as bundle:
        bundle.extractall(lab_root)
    os.chdir(lab_root)
print("Working directory:", Path.cwd())
assert Path("unified/train.py").is_file()


## 2. Install and check the environment
This uses the notebook's existing Python environment. If CUDA is unavailable after installation, check the GPU runtime and use the [official PyTorch installer](https://pytorch.org/get-started/locally/) for your machine.


In [ ]:
%pip install -r requirements.txt


In [ ]:
import torch
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name())
else:
    print("The tests below run on CPU. Select a CUDA GPU runtime before the training cells.")


## 3. Check the building blocks
Expected: four tests end in `OK`. Read `digit_model.py`, then `unified/processes.py`, then the loss/optimizer section of `unified/train.py`. For flow matching, explain why the target is `data - noise`.


In [ ]:
import subprocess
import sys

def run_lab(*arguments):
    with subprocess.Popen([sys.executable, *arguments], stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True, bufsize=1) as process:
        for line in process.stdout:
            print(line, end="", flush=True)
        if process.wait():
            raise subprocess.CalledProcessError(process.returncode, process.args)

run_lab("-m", "unittest", "unified.test_unified", "-v")


## 4. Run the 200-update smoke experiment
This checks the pipeline. It is too short to judge generation quality and ends before the standard warmup completes. Expected files: `results/smoke/flow/checkpoint.pt`, `metadata.json`, and `loss.npy`. MNIST downloads on first use.


In [ ]:
assert torch.cuda.is_available(), "Select a CUDA GPU runtime before training."
run_lab("-m", "unified.train", "--stage", "flow", "--steps", "200",
        "--batch-size", "32", "--root", "results/smoke")
run_lab("-m", "unified.sample", "--checkpoint", "results/smoke/flow/checkpoint.pt",
        "--method", "fm", "--nfe", "16", "--out", "outputs/smoke-flow.png")


In [ ]:
from IPython.display import display, Image
# Rows request digits 0–9. Expect poor samples from this short run.
display(Image(filename="outputs/smoke-flow.png"))


## 5. Train the full base flow
This uses a fresh directory and the 20,000-update, batch-128 base budget. Runtime depends on the GPU. Progress prints every 500 updates; checkpoints save every 2,000 updates and at normal completion.

If interrupted, rerun the command with `--resume` and keep `--steps 20000`. Without a saved checkpoint, restart the stage. The short run above stays separate.


In [ ]:
run_lab("-m", "unified.train", "--stage", "flow", "--steps", "20000", "--root", "results/student")


In [ ]:
run_lab("-m", "unified.sample", "--checkpoint", "results/student/flow/checkpoint.pt",
        "--method", "fm", "--nfe", "64", "--out", "outputs/flow-64.png", "--gif")
display(Image(filename="outputs/flow-64.png"))
display(Image(filename="outputs/flow-64.gif"))


## 6. Change sampling cost, keeping the model fixed
Generate at 1, 4, and 16 calls with the same seed and checkpoint. Compare both digit recognizability and variation within each row with the 64-call output. Write down what changes.


In [ ]:
for nfe in [1, 4, 16]:
    run_lab("-m", "unified.sample", "--checkpoint", "results/student/flow/checkpoint.pt",
            "--method", "fm", "--nfe", str(nfe), "--out", f"outputs/flow-{nfe}.png")
    display(Image(filename=f"outputs/flow-{nfe}.png"))


## 7. Optional: diffusion, reflow, and quantitative evaluation
Continue with [steps 6–8 of the guide](https://clguo.github.io/tutorials/video-generation/foundations/start-here.html#6-train-diffusion-once-then-compare-two-samplers). The cell below is disabled by default so **Run all** does not launch every extra experiment. Set `run_full_comparison = True` when you have the GPU time and have completed the full base flow.

DDPM and DDIM share one trained denoiser. Reflow trains a new student using teacher pairs. Continued flow is the extra-training control. The full evaluator also requires an independent classifier.


In [ ]:
run_full_comparison = False
if run_full_comparison:
    run_lab("-m", "unified.train", "--stage", "diffusion", "--steps", "20000", "--root", "results/student")
    for method in ["ddpm", "ddim"]:
        run_lab("-m", "unified.sample", "--checkpoint", "results/student/diffusion/checkpoint.pt",
                "--method", method, "--nfe", "64", "--out", f"outputs/{method}-64.png")
        display(Image(filename=f"outputs/{method}-64.png"))
    run_lab("-m", "unified.train", "--stage", "pairs", "--pairs", "20000", "--teacher-nfe", "128", "--root", "results/student")
    for stage in ["reflow", "continued"]:
        run_lab("-m", "unified.train", "--stage", stage, "--steps", "12000", "--root", "results/student")
        method = "rf2" if stage == "reflow" else "continued"
        run_lab("-m", "unified.sample", "--checkpoint", f"results/student/{stage}/checkpoint.pt",
                "--method", method, "--nfe", "1", "--out", f"outputs/{method}-1.png")
        display(Image(filename=f"outputs/{method}-1.png"))
    run_lab("quality.py", "--train-classifier")
    run_lab("-m", "unified.evaluate", "--root", "results/student")
    print("Read results/student/evaluation/report.json together with the saved sample grids.")


## Keep your experiment notes
For each change, record the command, checkpoint hash, seed, training updates, sampling calls, output grid, and your interpretation. JSON files next to the PNGs contain the sampling provenance. All samples are kept; there is no classifier-based selection.

Before moving to DiT, explain how the input and target are constructed, what the network predicts, and how the sampler updates the state.
